# P3 classic MASCHInE transfer (tc01–tc15, excluding tc04)

`synthetic_compare.ipynb` introduced the **P3 protograph** (P1 + full `subClassOf` hierarchy) and evaluated `p3_bound`, but never ran the symmetric **`p3_classic`** baseline: P3 pretrain → own-class mean init (`all_init` fallback) → protected finetune at LR 0.0025.

This notebook reports that missing variant on **tc01–tc15 excluding tc04**, using the same recipe as `p1_classic` / `p2_classic` in `synthetic_compare.ipynb`. Raw numbers live in `notebooks/p3_classic/results.json` (generated by `scripts/_run_p3_classic.py`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
P3_JSON = ROOT / "notebooks" / "p3_classic" / "results.json"
COMPARE_JSON = ROOT / "notebooks" / "synthetic_compare" / "results.json"

TCS = [f"tc{i:02d}" for i in range(1, 16) if i != 4]
CLASSIC_VARIANTS = ["p1_classic", "p2_classic", "p3_classic"]
BOUND_VARIANTS = ["p1_bound", "p2_bound", "p3_bound"]
ALL_VARIANTS = ["vanilla"] + CLASSIC_VARIANTS + BOUND_VARIANTS

p3_results = json.loads(P3_JSON.read_text(encoding="utf-8"))
compare_raw = json.loads(COMPARE_JSON.read_text(encoding="utf-8")) if COMPARE_JSON.is_file() else {}


def row_for(tc: str, variant: str) -> dict | None:
    if variant == "p3_classic":
        return p3_results.get(tc)
    for row in compare_raw.get(tc, []):
        if row["variant"] == variant:
            return row
    return None


def final_table(variants: list[str]) -> pd.DataFrame:
    rows = {}
    for tc in TCS:
        rows[tc] = {
            v: (row_for(tc, v) or {}).get("final_acc", np.nan)
            for v in variants
        }
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.loc["mean"] = df.mean(numeric_only=True)
    return df


def init_table(variants: list[str]) -> pd.DataFrame:
    rows = {}
    for tc in TCS:
        rows[tc] = {}
        for v in variants:
            row = row_for(tc, v)
            rows[tc][v] = row["accs"][0] if row and row.get("accs") else np.nan
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.loc["mean"] = df.mean(numeric_only=True)
    return df


print(f"P3 classic TCs: {', '.join(TCS)}")
print(f"Loaded p3_classic for {len(p3_results)} TCs")

## Final test accuracy (epoch 5)

Classic trio side-by-side; `p3_bound` included as the concept-bound counterpart on P3 codes.

In [ ]:
classic_final = final_table(["vanilla"] + CLASSIC_VARIANTS + ["p3_bound"])
display(classic_final.style.format("{:.4f}").background_gradient(cmap="RdYlGn", axis=None, vmin=0.5, vmax=1.0))

delta = classic_final.sub(classic_final["vanilla"], axis=0)
delta = delta.drop(columns=["vanilla"])
print("\nΔ vs vanilla (accuracy points)")
display(delta.style.format("{:+.4f}"))

## Accuracy at initialization (epoch 0)

P3 pretrain puts every leaf class in the vocabulary, so class-mean inits are non-random even for deep instance types. Compare init accuracy across the three classic variants.

In [ ]:
init_acc = init_table(CLASSIC_VARIANTS + ["p3_bound"])
display(init_acc.style.format("{:.4f}").background_gradient(cmap="RdYlGn", axis=None, vmin=0.45, vmax=1.0))

## Full variant matrix (where cached in `synthetic_compare`)

`tc02` was not in the original `FOCUS_TCS` list — only `p3_classic` is available there.

In [ ]:
full = final_table(ALL_VARIANTS)
display(full.style.format("{:.4f}"))

## Finetune curves — classic variants

In [ ]:
COLORS = {
    "p1_classic": "#c5b3e6",
    "p2_classic": "#7e57c2",
    "p3_classic": "#4527a0",
    "p3_bound": "#0d47a1",
    "vanilla": "#9e9e9e",
}

fig, axes = plt.subplots(3, 5, figsize=(16, 9), sharey=True)
axes = axes.ravel()
epochs = np.arange(6)

for ax, tc in zip(axes, TCS):
    for variant in ["vanilla", "p1_classic", "p2_classic", "p3_classic", "p3_bound"]:
        row = row_for(tc, variant)
        if not row:
            continue
        ax.plot(epochs, row["accs"], marker="o", ms=3, lw=1.5, label=variant, color=COLORS[variant])
    ax.axhline(0.5, color="k", ls=":", lw=0.8, alpha=0.4)
    ax.set_title(tc, fontsize=10)
    ax.set_xticks(epochs)
    ax.set_xlim(-0.2, 5.2)
    ax.set_ylim(0.45, 1.02)

for ax in axes[len(TCS):]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.02), fontsize=9)
fig.supxlabel("epoch (0 = after init, before finetune)")
fig.supylabel("LogReg test accuracy")
fig.suptitle("p3_classic vs other variants — per-TC accuracy curves", y=1.06, fontsize=12)
plt.tight_layout()
plt.show()

## Init coverage (`class_init` token counts)

All instance tokens received a class-mean vector (no random fallbacks) on every TC.

In [ ]:
init_rows = []
for tc in TCS:
    row = p3_results[tc]
    stats = row["init_stats"]
    init_rows.append(
        dict(
            tc=tc,
            stage1_copied=stats["stage1_copied"],
            class_init=stats["class_init"],
            random=stats["random"],
            init_acc=row["accs"][0],
            final_acc=row["final_acc"],
            seconds=row["seconds"],
        )
    )

init_df = pd.DataFrame(init_rows).set_index("tc")
display(init_df.style.format({"init_acc": "{:.4f}", "final_acc": "{:.4f}", "seconds": "{:.1f}"}))

## Summary

| observation | detail |
|---|---|
| **Coverage fix** | P3 puts every class in the pretrain vocabulary; `random=0` on all TCs (vs frequent fallbacks under P1/P2 on tc07–tc12). |
| **tc14 sanity check** | `p3_classic` reaches **1.00** — own-class init is exact when the label *is* class membership. |
| **Relation-centric TCs (tc07–tc12)** | `p3_classic` **learns during finetune** (curves rise above chance), unlike `p1_classic` which stays flat at ≈0.5 under the protected LR. Final accuracy remains well below `p3_bound` because own-class init still cannot encode neighbour-class / cardinality signal. |
| **Existence / mixed TCs (tc01–tc06)** | `p3_classic` is competitive with `p2_classic` and often beats `p1_classic` (tc01: 0.93, tc06: 0.97). |
| **Stability TCs (tc13–tc15)** | `p3_classic` underperforms `p3_bound` (tc13: 0.53 vs 0.86; tc15: 0.57 vs 0.94) — hierarchy coverage alone does not substitute for concept-bound init on these tasks. |
| **Mean over 14 TCs** | final accuracy: `p1_classic` 0.601 → `p3_classic` 0.723 (+0.11 vs P1) ≈ `p2_classic` 0.735; `p3_bound` 0.901 remains best overall. |

**Conclusion.** `p3_classic` confirms that the P3 protograph improves *classic* MASCHInE transfer primarily through **vocabulary coverage** (every leaf class gets a meaningful code). That is enough for class-membership tasks and allows partial learning on relation-centric TCs, but it does not replace concept-bound initialization for tc07–tc12.